# Text2SQL 개요


- **Text2SQL**은 자연어 처리(NLP)와 데이터베이스 기술을 결합하여, 사용자가 일상적인 언어로 질문하면 이를 구조화된 SQL 쿼리로 자동 변환하는 기술입니다.

     ```
     질문: "삼성자산운용의 평균 수익률은 얼마인가요?"
     SQL: SELECT AVG(최근1년_수익률_퍼센트) FROM ETFsInfo WHERE 운용사 = '삼성자산운용';
     
     질문: "각 운용사의 ETF 수를 세어주세요."
     SQL: SELECT 운용사, COUNT(*) as etf_count FROM ETFsInfo GROUP BY 운용사;
     
     질문: "수익률이 가장 높은 ETF의 종목명은?"
     SQL: SELECT 종목명 FROM ETFsInfo ORDER BY 최근1년_수익률_퍼센트 DESC LIMIT 1;
     ```

- **Text2SQL의 핵심 구성요소**

     | 구성요소 | 설명 |
     |---------|------|
     | **자연어 이해 (NLU)** | 사용자의 질문 의도를 파악하고 핵심 엔티티 추출 |
     | **스키마 매핑** | 질문의 개념을 데이터베이스 테이블/컬럼에 매핑 |
     | **쿼리 생성** | 문법적으로 올바른 SQL 쿼리 구성 |
     | **결과 해석** | 쿼리 실행 결과를 사용자 친화적으로 변환 |


- **Text2SQL의 활용 사례**

     - **비즈니스 인텔리전스**: 비개발자도 데이터 분석 가능
     - **챗봇 인터페이스**: 대화형 데이터 조회 시스템
     - **리포트 자동화**: 자연어 요청으로 보고서 생성
     - **데이터 탐색**: 복잡한 쿼리 없이 데이터 인사이트 도출

- **LLM 기반 Text2SQL의 장점**

     1. **유연한 자연어 처리**: 다양한 표현 방식 이해
     2. **컨텍스트 인식**: 대화 맥락을 고려한 쿼리 생성
     3. **스키마 적응**: 새로운 데이터베이스에 빠르게 적응
     4. **오류 복구**: 잘못된 쿼리 자동 수정 가능

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob
from pprint import pprint
import json
from typing import TypedDict, Annotated, List, Optional

# LangGraph & LangChain imports
from langgraph.graph import StateGraph, END
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, PydanticOutputParser
from langchain_core.utils.function_calling import convert_to_openai_function
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field


/Users/jiyun/004_llm_agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## ETF 데이터 로드

- ETF 목록: CSV 다운로드 ( http://data.krx.co.kr/contents/MDC/MDI/mdiLoader/index.cmd?menuId=MDC020103010901 )
- ETF 상세 정보: 이전 단계에서 수집해서 저장한 CSV 문서를 로드

`(1) ETF 목록`

In [3]:
import pandas as pd
import numpy as np

# ETF 목록
etf_data = pd.read_csv('data/etf_list.csv', encoding='cp949')
etf_data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/etf_list.csv'

`(2) ETF 상세 정보`

In [ ]:
# ETF 상세 정보
etf_info = pd.read_csv('data/etf_info.csv', encoding='cp949')
etf_info.head(2)

## **Text2SQL** 

- 실습 데이터: ETF 목록 데이터 ('`data/etf_list.csv`')


### 1) **SQLite Database**에 저장

- ETF 데이터를 CSV에서 읽어 SQLite **데이터베이스 테이블** 생성

- **데이터 타입** 최적화: 종목코드(INTEGER PRIMARY KEY), 수익률/총보수(FLOAT), 문자열(TEXT)

- 기본 **통계 분석**: 전체 ETF 수, 운용사 수, 평균 수익률/총보수 등 산출

In [ ]:
# ETF 목록
etf_data = pd.read_csv('data/etf_list.csv', encoding='cp949')
etf_data.head(2)

In [ ]:
# 열 이름 변경
etf_data.columns = ['종목코드', '종목명', '상장일', '분류체계', '운용사', '최근1년_수익률_퍼센트', '기초지수', '추적오차_퍼센트',
       '순자산총액_원', '괴리율_퍼센트', '변동성', '복제방법', '총보수_퍼센트', '과세유형']

In [ ]:
import pandas as pd

# 데이터 타입 변환
def convert_to_numeric_safely(value):
    try:
        return pd.to_numeric(value)
    except:
        return None

# 각 컬럼의 데이터 타입 변환
etf_data['종목코드'] = etf_data['종목코드'].apply(lambda x: str(x).strip())
etf_data['최근1년_수익률_퍼센트'] = etf_data['최근1년_수익률_퍼센트'].apply(convert_to_numeric_safely)
etf_data['추적오차_퍼센트'] = etf_data['추적오차_퍼센트'].apply(convert_to_numeric_safely)
etf_data['순자산총액_원'] = etf_data['순자산총액_원'].apply(convert_to_numeric_safely)
etf_data['괴리율_퍼센트'] = etf_data['괴리율_퍼센트'].apply(convert_to_numeric_safely)
etf_data['총보수_퍼센트'] = etf_data['총보수_퍼센트'].apply(convert_to_numeric_safely)

# 문자열 데이터 정리
string_columns = ['종목명', '상장일', '분류체계', '운용사', '기초지수', '변동성', '복제방법', '과세유형']
for col in string_columns:
    etf_data[col] = etf_data[col].astype(str).apply(lambda x: x.strip())


# 데이터 타입 확인
etf_data.info()

In [ ]:
import pandas as pd
import sqlite3

# SQLite 데이터베이스 생성
conn = sqlite3.connect('etf_database.db')
cursor = conn.cursor()

# 테이블 삭제 (if exists)
cursor.execute("DROP TABLE IF EXISTS ETFs")

# 테이블 생성
cursor.execute("""
CREATE TABLE ETFs (
    종목코드 TEXT PRIMARY KEY,
    종목명 TEXT,
    상장일 TEXT,
    분류체계 TEXT,
    운용사 TEXT,
    최근1년_수익률_퍼센트 REAL,
    기초지수 TEXT,
    추적오차_퍼센트 REAL,
    순자산총액_원 INTEGER,
    괴리율_퍼센트 REAL,
    변동성 TEXT,
    복제방법 TEXT,
    총보수_퍼센트 REAL,
    과세유형 TEXT
)
""")

# 데이터 삽입
for _, row in etf_data.iterrows():
    try:
        cursor.execute("""
        INSERT INTO ETFs VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            str(row['종목코드']),
            str(row['종목명']),
            str(row['상장일']),
            str(row['분류체계']),
            str(row['운용사']),
            float(row['최근1년_수익률_퍼센트']) if pd.notna(row['최근1년_수익률_퍼센트']) else None,
            str(row['기초지수']),
            float(row['추적오차_퍼센트']) if pd.notna(row['추적오차_퍼센트']) else None,
            int(row['순자산총액_원']) if pd.notna(row['순자산총액_원']) else None,
            float(row['괴리율_퍼센트']) if pd.notna(row['괴리율_퍼센트']) else None,
            str(row['변동성']),
            str(row['복제방법']),
            float(row['총보수_퍼센트']) if pd.notna(row['총보수_퍼센트']) else None,
            str(row['과세유형'])
        ))
    except Exception as e:
        print(f"Error inserting row: {row}")
        print(f"Error message: {str(e)}")
        continue

# 변경사항 저장
conn.commit()

# 데이터베이스 상태 확인
cursor.execute("SELECT COUNT(*) FROM ETFs")
etf_count = cursor.fetchone()[0]
print(f"\n=== 데이터베이스 생성 완료 ===")
print(f"ETF 개수: {etf_count}")

In [ ]:
# 데이터 확인
print("\n=== 데이터 샘플 ===")
cursor.execute("SELECT * FROM ETFs LIMIT 1")
columns = [description[0] for description in cursor.description]
row = cursor.fetchone()
if row:
    for col, val in zip(columns, row):
        print(f"{col}: {val}")

In [ ]:
# 데이터베이스 종료
conn.close()

### 2) **LangChain**에 연동

- **LangChain**과 ETF DB 연동으로 자연어 쿼리 처리 가능

- **GPT**와 **Gemini** 모델을 활용한 SQL 쿼리 자동 생성

- 한국어 응답을 위한 **QA Chain** 구성 및 쿼리 실행 도구 설정

`(1) DB 스키마 확인`
   - 작업의 첫 단계로 테이블 목록 확인 필요
   - 각 테이블의 **구조와 관계** 파악을 위한 스키마 정보 검토

In [ ]:
from langchain_community.utilities import SQLDatabase

# SQLite 데이터베이스 연결
db = SQLDatabase.from_uri("sqlite:///etf_database.db")

# 사용 가능한 테이블 목록 출력
tables = db.get_usable_table_names()
print(tables)

In [ ]:
# 테이블 스키마 정보 출력
print(db.get_table_info())

In [ ]:
# 기본 쿼리 실행
query = "SELECT * FROM ETFs LIMIT 5"
result = db.run(query)
print(result)

`(2) SQL Toolkit 도구`

- SQLDatabaseToolkit는 SQL 데이터베이스와 상호작용하기 위한 4가지 핵심 도구를 제공합니다.

    - sql_db_query: SQL 쿼리를 실행하고 결과를 반환합니다.
    - sql_db_schema: 지정한 테이블의 스키마와 샘플 데이터를 반환합니다.
    - sql_db_list_tables: 데이터베이스의 모든 테이블 목록을 반환합니다.
    - sql_db_query_checker: 쿼리 실행 전에 구문 오류를 검사합니다.


In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.chat_models import init_chat_model

# LLM 초기화
llm = init_chat_model("gpt-4.1-mini", temperature=0)

# SQL Toolkit 생성
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# 사용 가능한 도구 확인
tools = toolkit.get_tools()

print("SQL Toolkit 도구 목록:\n")
for tool in tools:
    print(f"도구명: {tool.name}")
    print(f"설명: {tool.description}")
    print("-" * 60)

`(3) SQL Agent 생성`

- create_agent를 사용하여 자연어 질문을 SQL로 변환하는 Agent를 생성합니다.


In [ ]:
from langchain.agents import create_agent

# System Prompt 정의
system_prompt = """
당신은 SQL 데이터베이스와 상호작용하는 전문 Agent입니다.
사용자의 질문을 받으면 다음 절차를 따르세요:

1. 데이터베이스에 어떤 테이블이 있는지 확인합니다.
2. 관련 테이블의 스키마를 조회합니다.
3. 질문에 답하기 위한 SQL 쿼리를 작성합니다.
4. 쿼리를 실행하기 전에 반드시 검증합니다.
5. 쿼리를 실행하고 결과를 해석하여 답변합니다.

중요한 규칙:
- 사용자가 특정 개수를 요청하지 않으면 최대 {top_k}개의 결과만 반환하세요.
- SELECT *를 사용하지 말고, 필요한 컬럼만 명시하세요.
- INSERT, UPDATE, DELETE, DROP 등의 쿼리는 절대 실행하지 마세요.
- 쿼리 실행 중 오류가 발생하면 쿼리를 수정하여 재시도하세요.
- 결과를 한글로 명확하게 설명하세요.

사용 중인 데이터베이스: {dialect}
""".format(dialect=db.dialect, top_k=5)

# Agent 생성
agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
)

In [ ]:
# 질문 1: 간단한 조회
response = agent.invoke({
    "messages": [{"role": "user", "content": "총자산 기준 상위 5개 운용사의 ETF 개수는 각각 몇 개인가요?"}]
})

print(response['messages'][-1].content)

In [ ]:
response['messages']

`(4) Few-shot 예제 추가`

- Agent에게 예제를 제공하여 더 정확한 쿼리를 생성하도록 할 수 있습니다.


In [ ]:
# Few-shot 예제를 포함한 System Prompt
few_shot_system_prompt = """
당신은 SQL 데이터베이스와 상호작용하는 전문 Agent입니다.

다음은 질문과 SQL 쿼리의 예제입니다:

질문: "삼성자산운용의 평균 수익률은 얼마인가요?"
SQL: SELECT AVG(최근1년_수익률_퍼센트) FROM ETFs WHERE 운용사 = '삼성자산운용';

질문: "각 운용사의 ETF 수를 세어주세요."
SQL: SELECT 운용사, COUNT(*) as etf_count FROM ETFs GROUP BY 운용사;

질문: "수익률이 가장 높은 ETF의 종목명은?"
SQL: SELECT 종목명 FROM ETFs ORDER BY 최근1년_수익률_퍼센트 DESC LIMIT 1;

이제 사용자의 질문에 답하세요.
사용 중인 데이터베이스: {dialect}
최대 결과 수: {top_k}
""".format(dialect=db.dialect, top_k=5)

# Few-shot Agent 생성
few_shot_agent = create_agent(
    llm,
    tools,
    system_prompt=few_shot_system_prompt,
)

# 테스트
response = few_shot_agent.invoke({
    "messages": [{"role": "user", "content": "자산운용사별로 가장 수익률이 좋은 ETF를 찾아주세요."}]
})

print("질문: 자산운용사별로 가장 수익률이 좋은 ETF를 찾아주세요.")
print(f"답변: {response['messages'][-1].content}")


---

## [실습] **ETF 상세정보를 Text2SQL 구현**

- data/etf_info.csv 데이터를 SQLite 스키마를 정의하여 "ETFInfo" 테이블에 저장
- SQL 쿼리 체인을 실행하여 테스트 

In [ ]:
etf_info.head(2)

In [ ]:
# 컬럼명 변경 
etf_info.columns = ['한글명', '영문명', '종목코드', '상장일', '펀드형태', '기초지수명', '추적배수', '자산운용사', 
       '지정참가회사_AP', '총보수_퍼센트', '회계기간', '과세유형', '분배금지급일', '홈페이지', '기초시장',
       '기초자산', '기본정보', '투자유의사항']

In [ ]:
# 자료형 확인 
etf_info.info()

In [ ]:
# 여기에 코드를 작성하세요. 

<details>
<summary>예시 정답</summary>

```python
def create_etfs_info_table(conn, etf_data):
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS ETFsInfo")  # 테이블명: ETFsInfo -> 기존에 존재하면 삭제
    
    cursor.execute("""
    CREATE TABLE ETFsInfo (
        한글명 TEXT,
        영문명 TEXT,
        종목코드 TEXT PRIMARY KEY,
        상장일 TEXT,
        펀드형태 TEXT,
        기초지수명 TEXT,
        추적배수 TEXT,
        자산운용사 TEXT,
        지정참가회사_AP TEXT,
        총보수_퍼센트 REAL,
        회계기간 TEXT,
        과세유형 TEXT,
        분배금지급일 TEXT,
        홈페이지 TEXT,
        기초시장 TEXT,
        기초자산 TEXT,
        기본정보 TEXT,
        투자유의사항 TEXT
    )
    """)
    

    # 데이터 삽입
    for _, row in etf_data.iterrows():
        try:
            cursor.execute("""
            INSERT INTO ETFsInfo VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                str(row['한글명']),
                str(row['영문명']),
                str(row['종목코드']),
                str(row['상장일']),
                str(row['펀드형태']),
                str(row['기초지수명']),
                str(row['추적배수']),
                str(row['자산운용사']),
                str(row['지정참가회사_AP']),
                float(row['총보수_퍼센트']) if pd.notna(row['총보수_퍼센트']) else None,
                str(row['회계기간']),
                str(row['과세유형']),
                str(row['분배금지급일']),
                str(row['홈페이지']),
                str(row['기초시장']),
                str(row['기초자산']),
                str(row['기본정보']),
                str(row['투자유의사항'])
            ))
        except Exception as e:
            print(f"오류 - 종목코드: {row['종목코드']}")
            print(f"상세: {str(e)}")
            continue


    conn.commit()
    cursor.execute("SELECT COUNT(*) FROM ETFsInfo")
    print(f"\n=== ETFsInfo 테이블 생성 완료 (총 {cursor.fetchone()[0]}개) ===")
    return conn


# 데이터베이스 연결 및 테이블 생성
conn = sqlite3.connect('etf_database.db')
conn = create_etfs_info_table(conn, etf_info)

# 데이터베이스 종료 
conn.close()

from langchain_community.utilities import SQLDatabase

# ETFs 테이블 제외
db = SQLDatabase.from_uri(
    "sqlite:///etf_database.db",
    ignore_tables=["ETFs"],
    )

from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain.chat_models import init_chat_model

# LLM 초기화
llm = init_chat_model("gpt-4.1-mini", temperature=0)

# SQL Toolkit 생성
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# 사용 가능한 도구 확인
tools = toolkit.get_tools()

from langchain.agents import create_agent

# System Prompt 정의
system_prompt = """
당신은 SQL 데이터베이스와 상호작용하는 전문 Agent입니다.
사용자의 질문을 받으면 다음 절차를 따르세요:

1. 데이터베이스에 어떤 테이블이 있는지 확인합니다.
2. 관련 테이블의 스키마를 조회합니다.
3. 질문에 답하기 위한 SQL 쿼리를 작성합니다.
4. 쿼리를 실행하기 전에 반드시 검증합니다.
5. 쿼리를 실행하고 결과를 해석하여 답변합니다.

중요한 규칙:
- 사용자가 특정 개수를 요청하지 않으면 최대 {top_k}개의 결과만 반환하세요.
- SELECT *를 사용하지 말고, 필요한 컬럼만 명시하세요.
- INSERT, UPDATE, DELETE, DROP 등의 쿼리는 절대 실행하지 마세요.
- 쿼리 실행 중 오류가 발생하면 쿼리를 수정하여 재시도하세요.
- 결과를 한글로 명확하게 설명하세요.

사용 중인 데이터베이스: {dialect}
""".format(dialect=db.dialect, top_k=5)

# Agent 생성
agent = create_agent(
    llm,
    tools,
    system_prompt=system_prompt,
)


# 테스트
response = agent.invoke({
    "messages": [{"role": "user", "content": "총보수가 높은 상위 5개 ETF를 알려주세요."}]
})

print(response['messages'][-1].content)
```

</details>